# Part 1 Demo — TUDTK

Notebook này trình bày và kiểm thử các thuật toán **Phần 1**:

| Module | Chức năng |
|---|---|
| `gaussian.py` | Khử Gauss, thế ngược |
| `determinant.py` | Tính định thức |
| `inverse.py` | Tính ma trận nghịch đảo |
| `rank_basis.py` | Tính hạng & không gian cơ sở |


## 1. Import thư viện & các module

In [1]:
import numpy as np
import random

from gaussian import gaussian_eliminate, back_substitution
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis

print("✅ Import thành công!")


✅ Import thành công!


## 2. Các hàm kiểm thử (Verify)

Mỗi hàm `verify_*` so sánh kết quả tự cài đặt với thư viện **NumPy** làm chuẩn.


In [2]:
def verify_solve(A, b, x_custom):
    """So sánh nghiệm Ax = b với numpy.linalg.solve."""
    if isinstance(x_custom, str) or (
        isinstance(x_custom, list) and isinstance(x_custom[0], str)
    ):
        print("   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]")
        return True
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float)
    x_np = np.array(x_custom, dtype=float)
    return np.allclose(A_np @ x_np, b_np)

def verify_determinant(A, det_custom):
    """So sánh định thức với numpy.linalg.det."""
    A_np = np.array(A, dtype=float)
    return np.isclose(det_custom, np.linalg.det(A_np))

def verify_inverse(A, inv_custom):
    """So sánh nghịch đảo: A · A⁻¹ ≈ I."""
    if isinstance(inv_custom, str):
        print("   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]")
        return True
    A_np   = np.array(A, dtype=float)
    inv_np = np.array(inv_custom, dtype=float)
    return np.allclose(A_np @ inv_np, np.eye(A_np.shape[0]))

def verify_rank(A, rank_custom):
    """So sánh rank với numpy.linalg.matrix_rank."""
    A_np = np.array(A, dtype=float)
    return rank_custom == np.linalg.matrix_rank(A_np)

print("✅ Các hàm verify đã sẵn sàng.")


✅ Các hàm verify đã sẵn sàng.


## 3. Hàm sinh dữ liệu & chạy test

In [3]:
def generate_random_test(n):
    """Sinh ma trận ngẫu nhiên n×n và vector b."""
    A = [[random.randint(-5, 5) for _ in range(n)] for _ in range(n)]
    b = [random.randint(-10, 10) for _ in range(n)]
    return A, b

def run_case(tc):
    """Chạy một test case và in kết quả."""
    A, b = tc["A"], tc["b"]
    is_large = tc.get("large", False)
    if not is_large:
        print(f"   A = {A}")
        print(f"   b = {b}")
    else:
        print(f"   (Ma trận lớn {len(A)}×{len(A[0])} — không hiển thị)")

    try:
        my_det  = determinant(A)
        my_rank = rank_and_basis(A)[0]
        _, my_x, _ = gaussian_eliminate(A, b)
        my_inv  = inverse(A)

        errors = []
        if not verify_solve(A, b, my_x):     errors.append("❌ Sai nghiệm Ax=b")
        if not verify_determinant(A, my_det): errors.append("❌ Sai định thức")
        if not verify_inverse(A, my_inv):     errors.append("❌ Sai nghịch đảo")
        if not verify_rank(A, my_rank):       errors.append("❌ Sai rank")

        if not errors:
            print("   ✅ Passed")
        else:
            print("   " + "  ".join(errors))
    except Exception as e:
        print(f"   💥 CRASH: {e}")

print("✅ Helper functions ready.")


✅ Helper functions ready.


## 4. Kiểm thử: Hệ phương trình Ax = b

### 4.1 Hệ có nghiệm duy nhất

Ma trận khả nghịch, pivot khác 0 ở mỗi cột → khử Gauss cho nghiệm duy nhất.


In [4]:
tc = {
    "name": "Hệ có nghiệm duy nhất",
    "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]],
    "b": [8, -11, -3]
}
print(f"📌 {tc['name']}")
run_case(tc)


📌 Hệ có nghiệm duy nhất
   A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
   b = [8, -11, -3]
   ✅ Passed


### 4.2 Pivot = 0 (cần hoán đổi dòng)

Phần tử đầu tiên của cột 0 bằng 0 → thuật toán phải chọn pivot theo partial pivoting.


In [5]:
tc = {
    "name": "Pivot = 0 (hoán đổi dòng)",
    "A": [[0, 2, 1], [1, -2, -3], [-1, 1, 2]],
    "b": [-8, 0, 3]
}
print(f"📌 {tc['name']}")
run_case(tc)


📌 Pivot = 0 (hoán đổi dòng)
   A = [[0, 2, 1], [1, -2, -3], [-1, 1, 2]]
   b = [-8, 0, 3]
   ✅ Passed


### 4.3 Hệ vô số nghiệm

Các dòng của ma trận tỉ lệ với nhau → hệ thiếu rank → xuất hiện ẩn tự do.


In [6]:
tc = {
    "name": "Vô số nghiệm",
    "A": [[1, 1, 1], [2, 2, 2], [3, 3, 3]],
    "b": [3, 6, 9]
}
print(f"📌 {tc['name']}")
run_case(tc)


📌 Vô số nghiệm
   A = [[1, 1, 1], [2, 2, 2], [3, 3, 3]]
   b = [3, 6, 9]
Không có pivot tại cột 1.
Không có pivot tại cột 2.
Không có pivot tại cột 1.
Không có pivot tại cột 2.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
   ✅ Passed


### 4.4 Hệ vô nghiệm

Hai dòng giống nhau nhưng vế phải khác nhau → mâu thuẫn → hệ vô nghiệm.


In [7]:
tc = {
    "name": "Vô nghiệm",
    "A": [[1, 1], [1, 1]],
    "b": [1, 2]
}
print(f"📌 {tc['name']}")
run_case(tc)


📌 Vô nghiệm
   A = [[1, 1], [1, 1]]
   b = [1, 2]
Không có pivot tại cột 1.
Không có pivot tại cột 1.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
   ✅ Passed


### 4.5 Các trường hợp đặc biệt khác

- **Suy biến**: ma trận có hàng phụ thuộc tuyến tính.  
- **Ma trận đơn vị**: kết quả x = b.  
- **Zero matrix**: toàn 0 → ẩn tự do hoàn toàn.  
- **Gần suy biến**: pivot rất nhỏ → cần partial pivoting ổn định.


In [8]:
special_cases = [
    {"name": "Suy biến",        "A": [[1, 2], [2, 4]],        "b": [3, 6]},
    {"name": "Ma trận đơn vị",  "A": [[1,0,0],[0,1,0],[0,0,1]], "b": [5,-3,2]},
    {"name": "Zero matrix",     "A": [[0, 0], [0, 0]],         "b": [0, 0]},
    {"name": "Gần suy biến",
     "A": [[1,1,1],[1,1.0000001,1],[1,1,1.0000001]],
     "b": [3, 3.0000001, 3.0000001]},
    {"name": "4x4 đầy đủ",
     "A": [[1,2,3,4],[2,5,2,1],[3,2,6,2],[4,1,2,7]], "b": [10,8,9,11]},
    {"name": "Rank < n",        "A": [[1,2,3],[2,4,6],[0,0,0]], "b": [6,12,0]},
]
for tc in special_cases:
    print(f"\n📌 {tc['name']}")
    run_case(tc)



📌 Suy biến
   A = [[1, 2], [2, 4]]
   b = [3, 6]
Không có pivot tại cột 1.
Không có pivot tại cột 1.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
   ✅ Passed

📌 Ma trận đơn vị
   A = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
   b = [5, -3, 2]
   ✅ Passed

📌 Zero matrix
   A = [[0, 0], [0, 0]]
   b = [0, 0]
Không có pivot tại cột 0.
Không có pivot tại cột 1.
Không có pivot tại cột 0.
Không có pivot tại cột 1.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
   ✅ Passed

📌 Gần suy biến
   A = [[1, 1, 1], [1, 1.0000001, 1], [1, 1, 1.0000001]]
   b = [3, 3.0000001, 3.0000001]
   ✅ Passed

📌 4x4 đầy đủ
   A = [[1, 2, 3, 4], [2, 5, 2, 1], [3, 2, 6, 2], [4, 1, 2, 7]]
   b = [10, 8, 9, 11]
   ✅ Passed

📌 Rank < n
   A = [[1, 2, 3], [2, 4, 6], [0, 0, 0]]
   b = [6, 12, 0]
Không có pivot tại cột 1.
Không có pivot tại cột 2.
Không có pivot tại cột 1.
Không có pivot tại cột 

### 4.6 Test hiệu năng — Ma trận lớn ngẫu nhiên

Kiểm tra tính chính xác và khả năng xử lý với kích thước lớn (30×30 đến 1000×1000).


In [9]:
import time

for size in [30, 50, 80, 100, 200, 500, 1000]:
    A, b = generate_random_test(size)
    tc = {"name": f"Random {size}×{size}", "A": A, "b": b, "large": True}
    t0 = time.time()
    print(f"\n📌 {tc['name']}")
    run_case(tc)
    print(f"   ⏱  {time.time()-t0:.3f}s")



📌 Random 30×30
   (Ma trận lớn 30×30 — không hiển thị)
   ✅ Passed
   ⏱  0.007s

📌 Random 50×50
   (Ma trận lớn 50×50 — không hiển thị)
   ✅ Passed
   ⏱  0.026s

📌 Random 80×80
   (Ma trận lớn 80×80 — không hiển thị)
   ✅ Passed
   ⏱  0.095s

📌 Random 100×100
   (Ma trận lớn 100×100 — không hiển thị)
   ✅ Passed
   ⏱  0.211s

📌 Random 200×200
   (Ma trận lớn 200×200 — không hiển thị)
   ✅ Passed
   ⏱  1.436s

📌 Random 500×500
   (Ma trận lớn 500×500 — không hiển thị)


C:\Users\Khoa\AppData\Roaming\Python\Python313\site-packages\numpy\linalg\_linalg.py:2383: RuntimeWarning: overflow encountered in det
  r = _umath_linalg.det(a, signature=signature)


   ✅ Passed
   ⏱  35.857s

📌 Random 1000×1000
   (Ma trận lớn 1000×1000 — không hiển thị)
   ✅ Passed
   ⏱  206.814s


## 5. Kiểm thử: Định thức

Định thức được tính theo công thức:

$$\det(A) = (-1)^{\text{swaps}} \times \prod_{i} U_{ii}$$

trong đó $U$ là ma trận tam giác trên thu được từ khử Gauss.


In [10]:
det_cases = [
    {"name": "3x3 cơ bản",       "A": [[2,1,-1],[-3,-1,2],[-2,1,2]]},
    {"name": "Ma trận đơn vị",   "A": [[1,0,0],[0,1,0],[0,0,1]]},
    {"name": "Ma trận suy biến", "A": [[1,2],[2,4]]},
    {"name": "4x4",              "A": [[1,2,3,4],[2,5,2,1],[3,2,6,2],[4,1,2,7]]},
    {"name": "2x2 âm",           "A": [[-3,1],[2,-4]]},
]

print(f"{'Tên':<25} {'det (custom)':>15} {'det (numpy)':>13} {'Kết quả':>10}")
print("-" * 68)
for tc in det_cases:
    A = tc["A"]
    my_det   = determinant(A)
    np_det   = np.linalg.det(np.array(A, dtype=float))
    ok       = "✅" if verify_determinant(A, my_det) else "❌"
    print(f"{tc['name']:<25} {my_det:>15.6f} {np_det:>13.6f} {ok:>10}")


Tên                          det (custom)   det (numpy)    Kết quả
--------------------------------------------------------------------
3x3 cơ bản                      -1.000000     -1.000000          ✅
Ma trận đơn vị                   1.000000      1.000000          ✅
Không có pivot tại cột 1.
Ma trận suy biến                -0.000000      0.000000          ✅
4x4                           -342.000000   -342.000000          ✅
2x2 âm                          10.000000     10.000000          ✅


## 6. Kiểm thử: Ma trận nghịch đảo

Nghịch đảo $A^{-1}$ được tính bằng cách giải $A \mathbf{x}_i = \mathbf{e}_i$  
cho từng vector đơn vị $\mathbf{e}_i$ rồi ghép các nghiệm thành cột.

Điều kiện kiểm tra: $A \cdot A^{-1} \approx I$.


In [11]:
inv_cases = [
    {"name": "3x3 khả nghịch",   "A": [[2,1,-1],[-3,-1,2],[-2,1,2]]},
    {"name": "Ma trận đơn vị",   "A": [[1,0,0],[0,1,0],[0,0,1]]},
    {"name": "4x4 khả nghịch",   "A": [[1,2,3,4],[2,5,2,1],[3,2,6,2],[4,1,2,7]]},
    {"name": "Ma trận suy biến (det=0)", "A": [[1,2],[2,4]]},
    {"name": "2x2 khả nghịch",   "A": [[4,7],[2,6]]},
]

for tc in inv_cases:
    A = tc["A"]
    my_inv = inverse(A)
    ok = verify_inverse(A, my_inv)
    status = "✅ Passed" if ok else "❌ Failed"
    print(f"📌 {tc['name']:<30} → {status}")
    if isinstance(my_inv, list):
        inv_np = np.array(my_inv)
        print(f"   A⁻¹ =\n{np.round(inv_np, 4)}")


📌 3x3 khả nghịch                 → ✅ Passed
   A⁻¹ =
[[ 4.  3. -1.]
 [-2. -2.  1.]
 [ 5.  4. -1.]]
📌 Ma trận đơn vị                 → ✅ Passed
   A⁻¹ =
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
📌 4x4 khả nghịch                 → ✅ Passed
   A⁻¹ =
[[-0.4795  0.0936  0.1345  0.2222]
 [ 0.0936  0.2135 -0.0994 -0.0556]
 [ 0.1345 -0.0994  0.1696 -0.1111]
 [ 0.2222 -0.0556 -0.1111  0.0556]]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
📌 Ma trận suy biến (det=0)       → ✅ Passed
📌 2x2 khả nghịch                 → ✅ Passed
   A⁻¹ =
[[ 0.6 -0.7]
 [-0.2  0.4]]


## 7. Kiểm thử: Rank & Không gian cơ sở

Hàm `rank_and_basis` trả về:
- **rank**: số chiều của không gian cột
- **column_space**: cơ sở của không gian cột
- **row_space**: cơ sở của không gian hàng
- **null_space**: cơ sở của không gian null (ker A)

Luôn thỏa mãn: $\text{rank}(A) + \dim(\ker A) = n$ (số cột).


In [12]:
rank_cases = [
    {"name": "Ma trận full rank",   "A": [[1,2,3],[0,1,4],[5,6,0]]},
    {"name": "Ma trận đơn vị 3x3",  "A": [[1,0,0],[0,1,0],[0,0,1]]},
    {"name": "Rank = 1",            "A": [[1,2,3],[2,4,6],[3,6,9]]},
    {"name": "Rank = 0 (zero)",     "A": [[0,0],[0,0]]},
    {"name": "4x4 rank đầy đủ",     "A": [[1,2,3,4],[2,5,2,1],[3,2,6,2],[4,1,2,7]]},
    {"name": "3x4 (m < n)",         "A": [[1,2,0,1],[0,1,1,0],[1,0,1,2]]},
]

print(f"{'Tên':<28} {'rank (custom)':>14} {'rank (numpy)':>13} {'Nullity':>8} {'Kết quả':>9}")
print("-" * 77)
for tc in rank_cases:
    A = tc["A"]
    my_rank, basis = rank_and_basis(A)
    np_rank  = np.linalg.matrix_rank(np.array(A, dtype=float))
    nullity  = len(A[0]) - my_rank
    ok       = "✅" if verify_rank(A, my_rank) else "❌"
    print(f"{tc['name']:<28} {my_rank:>14} {np_rank:>13} {nullity:>8} {ok:>9}")


Tên                           rank (custom)  rank (numpy)  Nullity   Kết quả
-----------------------------------------------------------------------------
Ma trận full rank                         3             3        0         ✅
Ma trận đơn vị 3x3                        3             3        0         ✅
Rank = 1                                  1             1        2         ✅
Rank = 0 (zero)                           0             0        2         ✅
4x4 rank đầy đủ                           4             4        0         ✅
3x4 (m < n)                               3             3        1         ✅


### 7.1 Kiểm tra chi tiết không gian Null

Xác nhận mọi vector trong null space thỏa $A \mathbf{v} = \mathbf{0}$.


In [13]:
A_test = [[1, 2, 3], [2, 4, 6], [3, 6, 9]]
_, basis = rank_and_basis(A_test)
null_vecs = basis["null_space"]

print(f"A = {A_test}")
print(f"Null space (số chiều = {len(null_vecs)}):")
A_np = np.array(A_test, dtype=float)
for i, v in enumerate(null_vecs):
    v_np = np.array(v)
    residual = np.linalg.norm(A_np @ v_np)
    print(f"  v{i+1} = {np.round(v_np, 4)},  ||A·v{i+1}|| = {residual:.2e}  {'✅' if residual < 1e-8 else '❌'}")


A = [[1, 2, 3], [2, 4, 6], [3, 6, 9]]
Null space (số chiều = 2):
  v1 = [-2.  1.  0.],  ||A·v1|| = 0.00e+00  ✅
  v2 = [-3.  0.  1.],  ||A·v2|| = 0.00e+00  ✅


## 8. Tổng kết — Bảng kết quả toàn bộ test cases

Chạy lại toàn bộ 10 test case chuẩn và in bảng tổng hợp.


In [14]:
all_cases = [
    {"name": "Nghiệm duy nhất",  "A": [[2,1,-1],[-3,-1,2],[-2,1,2]], "b": [8,-11,-3]},
    {"name": "Pivot = 0",        "A": [[0,2,1],[1,-2,-3],[-1,1,2]],  "b": [-8,0,3]},
    {"name": "Vô số nghiệm",     "A": [[1,1,1],[2,2,2],[3,3,3]],     "b": [3,6,9]},
    {"name": "Vô nghiệm",        "A": [[1,1],[1,1]],                  "b": [1,2]},
    {"name": "Suy biến",         "A": [[1,2],[2,4]],                  "b": [3,6]},
    {"name": "Ma trận đơn vị",   "A": [[1,0,0],[0,1,0],[0,0,1]],     "b": [5,-3,2]},
    {"name": "Zero matrix",      "A": [[0,0],[0,0]],                  "b": [0,0]},
    {"name": "Gần suy biến",
     "A": [[1,1,1],[1,1.0000001,1],[1,1,1.0000001]], "b": [3,3.0000001,3.0000001]},
    {"name": "4x4",
     "A": [[1,2,3,4],[2,5,2,1],[3,2,6,2],[4,1,2,7]], "b": [10,8,9,11]},
    {"name": "Rank < n",         "A": [[1,2,3],[2,4,6],[0,0,0]],     "b": [6,12,0]},
]

passed = 0
print(f"{'#':<3} {'Tên':<22} {'Gauss':>7} {'Det':>5} {'Inv':>5} {'Rank':>6} {'Tổng':>7}")
print("-" * 58)
for i, tc in enumerate(all_cases):
    A, b = tc["A"], tc["b"]
    try:
        my_det  = determinant(A)
        my_rank = rank_and_basis(A)[0]
        _, my_x, _ = gaussian_eliminate(A, b)
        my_inv  = inverse(A)
        r = [
            verify_solve(A, b, my_x),
            verify_determinant(A, my_det),
            verify_inverse(A, my_inv),
            verify_rank(A, my_rank),
        ]
        ok = all(r)
        if ok: passed += 1
        row = ["✅" if x else "❌" for x in r]
        total = "✅ OK" if ok else "❌ FAIL"
        print(f"{i+1:<3} {tc['name']:<22} {row[0]:>7} {row[1]:>5} {row[2]:>5} {row[3]:>6} {total:>7}")
    except Exception as e:
        print(f"{i+1:<3} {tc['name']:<22} {'💥 CRASH: '+str(e)}")

print("-" * 58)
print(f"Kết quả: {passed}/{len(all_cases)} test cases PASSED")


#   Tên                      Gauss   Det   Inv   Rank    Tổng
----------------------------------------------------------
1   Nghiệm duy nhất              ✅     ✅     ✅      ✅    ✅ OK
2   Pivot = 0                    ✅     ✅     ✅      ✅    ✅ OK
Không có pivot tại cột 1.
Không có pivot tại cột 2.
Không có pivot tại cột 1.
Không có pivot tại cột 2.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
3   Vô số nghiệm                 ✅     ✅     ✅      ✅    ✅ OK
Không có pivot tại cột 1.
Không có pivot tại cột 1.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
4   Vô nghiệm                    ✅     ✅     ✅      ✅    ✅ OK
Không có pivot tại cột 1.
Không có pivot tại cột 1.
   >> Nghiệm: [Hệ vô nghiệm/vô số nghiệm — bỏ qua so sánh]
   >> Nghịch đảo: [Ma trận suy biến — bỏ qua so sánh]
5   Suy biến                     ✅     ✅     ✅      ✅    ✅ OK
6   Ma trận đơn vị    